# Learning Objectives

* Build a fan-in Pegasus workflow from prebuilt radar wind tools in an Apptainer container.
* Generate and register input data, a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the maximum-wind field and the hospital alert produced by a run.

# CASA Wind Workflow

This workflow runs CASA (Collaborative Adaptive Sensing of the Atmosphere) damaging-wind detection over a
window of radar sweeps. It computes the maximum wind velocity across the window, renders it, vectorizes it to
GeoJSON, and intersects it with hospital locations to produce an alert. See `README.md` in this repository for
the full description.

**Workflow Jobs (per window):**
1. `gunzip` - decompress each radar sweep (one job per file)
2. `um_vel` - maximum wind velocity over the window
3. `merged_netcdf2png` - render the velocity field to a PNG
4. `mvt` - vectorize the velocity field to GeoJSON
5. `pointalert` - intersect it with hospital locations into an alert GeoJSON

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Sample Input Data

Five consecutive sweeps from the `addison.tx` CASA radar (2017-03-29, 07:10–07:14 UTC, the DFW hail and
wind event) ship in `input/`, with the hospital locations, the `pointAlert` config, and the wind color scale.
The last sweep's timestamp names every output.

In [ ]:
!ls -l input/

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/CASA_Wind_Container.def`) converted from the published
`pegasus/casa-wind` Docker image, which ships the tools prebuilt. Pegasus stages the `.sif` to the workers, so jobs need
no registry access. Building it can take a few minutes; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/CASA_Wind_Container.sif ]; then
    apptainer build Apptainer/CASA_Wind_Container.sif Apptainer/CASA_Wind_Container.def
else
    echo "Container already built: Apptainer/CASA_Wind_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `CASAWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs, embedded in
`workflow.yml`) and writes it with `pegasus.properties`. Nothing about the workflow is defined here: change the
pipeline in `workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` adds a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import CASAWorkflow, resolve_radar_file

EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

RADAR_FILES = sorted(resolve_radar_file(str(p)) for p in Path("input").glob("addison.tx-*.netcdf.gz"))
print(f"Input files: {[Path(f).name for f in RADAR_FILES]}")

workflow = CASAWorkflow(outdir=str(Path(".").resolve()), radar_files=RADAR_FILES)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating CASA Wind workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nCASA Wind Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. Five `gunzip` jobs fan in to one `um_vel` job, whose velocity field feeds the PNG render and the `mvt` → `pointalert` chain.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/` (with `HOSTED_SITE_CATALOG` set, Pegasus's
default `wf-output/` instead):

| Output | Description |
|--------|-------------|
| `MaxVelocity_{YYYYMMDD}-{HHMMSS}.netcdf` | Maximum wind velocity over the window |
| `MaxVelocity_{YYYYMMDD}-{HHMMSS}.png` | Rendered velocity field |
| `mvt_MaxVelocity_{YYYYMMDD}-{HHMMSS}.geojson` | Vectorized velocity field |
| `alert_{YYYYMMDD}-{HHMMSS}.geojson` | Hospitals inside the damaging-wind area |

In [ ]:
!ls -lt output/

The maximum wind velocity over the window, and the resulting hospital alert:

In [ ]:
Image(filename="output/MaxVelocity_20170329-071403.png")

In [ ]:
!cat output/alert_20170329-071403.geojson